In [145]:
import torch as t
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline
import yaml
import random
import itertools
import json
device = t.device("cuda" if t.cuda.is_available() else "cpu")
token = ""

In [2]:
model_id = "meta-llama/Llama-3.2-1B-Instruct"

In [3]:
text_generation_pipeline = pipeline(task="text-generation", model=model_id, device=device, token=token)

Device set to use cpu


In [73]:
with open('../configs/personas.yaml', 'r') as file:
    persona_config = yaml.safe_load(file)
    
with open('../configs/questions.yaml', 'r') as file:
    question_list = yaml.safe_load(file)

In [74]:
base_text = persona_config['customer_service']['base_text']
job_persona = persona_config['customer_service']['personas'][0]

In [7]:
def persona_curation(base_text, job_persona):
    return f"""{base_text} who is a {job_persona['Summary']}. You are {job_persona['Age']} years old, based out of {job_persona['Location']}.
You have a background as a {job_persona['Background']}, you are {job_persona['Personality Traits']}
and {job_persona['Style']}"""

def prompt_formatting(persona, base_prompt, questions):
    prompt = f"{persona} \n Question: {base_prompt} \n {questions}"
    return prompt

In [8]:
base_prompt = """Answer the questions on a scale of 1-5, varying from strongly disagree to strongly agree. 
Do not give any reasonings and directly answer with the degree of agreement or disagreement."""

questions = "\n ".join([f"{str(i+1)}. {question}" for i, question in enumerate(question_list)])

In [10]:
final_prompt = prompt_formatting(persona_curation(base_text, job_persona), base_prompt, questions)
print(final_prompt)

You are a customer service professional. who is a Calm Crisis Handler. You are 42 years old, based out of Lagos, Nigeria.
You have a background as a Former emergency dispatcher, you are Calm under pressure, reassuring, composed
and Handles escalations and crisis calls with grace. Keeps things clear and controlled. 
 Question: Answer the questions on a scale of 1-5, varying from strongly disagree to strongly agree. 
Do not give any reasonings and directly answer with the degree of agreement or disagreement. 
 1. I would be quite bored by a visit to an art gallery.
 2. I clean my office or home quite frequently.
 3. I rarely hold a grudge, even against people who have badly wronged me.


In [11]:
pipeline_config = {
    "temperature": 1,
    "num_return_sequences": 1,
}

generation_config = {
    "generation_type": "one_at_time",
    "shuffle":False,
    "reasoning": False,
    "reasoning_base_prompt": """Answer the questions on a scale of 1-5, varying from strongly disagree to strongly agree. 
Give your reasoning for choosing the answer along with the degree of agreement or disagreement.""",
    "non_reasoning_base_prompt": """Answer the questions on a scale of 1-5, varying from strongly disagree to strongly agree. 
Do not give any reasonings and directly answer with the degree of agreement or disagreement."""
}

In [128]:
def text_generate(prompts, pipeline, pipeline_config, generation_config):
    if generation_config["shuffle"]:
        shuffled_prompts = prompts.copy()
        random.shuffle(prompts)
        indices = [prompts.index(item) for item in shuffled_prompts]
    else:
        indices = list(range(len(prompts)))
    
    if generation_config["generation_type"] == "one_at_time":
        outputs = []
        for prompt in prompts:
            outputs.append(pipeline(prompt, **pipeline_config))
    else:
        outputs = pipeline(prompts, **pipeline_config)
        
    output_dict = {
        "indices": indices,
        "outputs": outputs
    }
    
    return output_dict
        
def generate_combinations(dict1, dict2):
        keys1, values1 = zip(*dict1.items())
        keys2, values2 = zip(*dict2.items())
        
        return [
            ({k: v for k, v in zip(keys1, combo1)}, 
             {k: v for k, v in zip(keys2, combo2)})
            for combo1 in itertools.product(*values1)
            for combo2 in itertools.product(*values2)
        ]

In [130]:
def experiment_setup(job_title, question_list, config_combinations, pipeline, base_prompt_config):
    
    experiment_results = {}
    print(f"Running Experiment for: {job_title}")
    base_text = persona_config[job_title]['base_text']
    personas = persona_config[job_title]['personas']
    
    print(f"Total no of config combinations: {len(config_combinations)}")
    print(f"Total no of personas: {len(personas)}")
    
    for persona in personas:
        print(f"Persona: {persona['Summary']}")
        persona_result = []
        for n, config in enumerate(config_combinations):
            print(f"Config No: {n}")
            pipeline_config, generation_config = config
            
            if generation_config['reasoning']:
                base_prompt = base_prompt_config["reasoning_base_prompt"]
            else:
                base_prompt = base_prompt_config["non_reasoning_base_prompt"]
                
            print(f"Generation Type: {generation_config['generation_type']}")
            
            if generation_config['generation_type'] == "one_at_time":
                prompts = []
                
                for question in question_list:
                    prompts.append(prompt_formatting(persona_curation(base_text, persona), base_prompt, question))
                
            else:
                questions = "\n ".join([f"{str(i+1)}. {question}" for i, question in enumerate(question_list)])
                prompts = prompt_formatting(persona_curation(base_text, persona), base_prompt, questions)
                
            
            text_generation_output_dict = text_generate(prompts,pipeline, pipeline_config, generation_config)
            
            text_generation_output_dict['generation_config'] = generation_config
            text_generation_output_dict['pipeline_config'] = pipeline_config
            
            persona_result.append(text_generation_output_dict)
        
        experiment_results[persona['Summary']] = persona_result
        
    return experiment_results

In [131]:
pipeline_config = {
    "temperature": [1,1.2,1.5],
    "num_return_sequences": [1,4,6]
}

base_prompt_config = {
    "reasoning_base_prompt": """Answer the questions on a scale of 1-5, varying from strongly disagree to strongly agree. 
Give your reasoning for choosing the answer along with the degree of agreement or disagreement.""",
    "non_reasoning_base_prompt": """Answer the questions on a scale of 1-5, varying from strongly disagree to strongly agree. 
Do not give any reasonings and directly answer with the degree of agreement or disagreement."""
}

generation_config = {
    "generation_type": ["one_at_time","all"],
    "shuffle":[False, True],
    "reasoning": [False, True]
}
config_combinations = generate_combinations(pipeline_config, generation_config)

In [132]:
job_title = "customer_service"
experiment_results = experiment_setup(job_title, question_list, config_combinations[:2], text_generation_pipeline, base_prompt_config)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Running Experiment for: customer_service
Total no of config combinations: 2
Total no of personas: 1
Persona: Calm Crisis Handler
Config No: 0
Generation Type: one_at_time


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Config No: 1
Generation Type: one_at_time


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [146]:
with open('experiment_results.json', 'w') as f:
    json.dump(experiment_results, f)